# belief eye neural analysis


## imports

In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:

import sys
import os
from pathlib import Path
import configparser
config = configparser.ConfigParser()
config.read_file(open('privateconfig'))
resdir = Path(config['Datafolder']['data'])
workdir = Path(config['Codefolder']['workspace'])
os.chdir(workdir)
# analysis
from scipy.io import loadmat
from sklearn.decomposition import FastICA
from sklearn.linear_model import LinearRegression
from sklearn.datasets import make_regression
from sklearn.model_selection import KFold
from sklearn.linear_model import LassoCV, Lasso
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr
from sklearn.cross_decomposition import CCA
# misc
import pickle
from collections import defaultdict
from neural_plot_ult import *
import time
tic=time.time()
import warnings
warnings.filterwarnings('ignore')

In [3]:
# run this once per notebook session  —  ideally in the first cell
from IPython import get_ipython

def notify_exc(shell, etype, evalue, tb, tb_offset=None):
    # 1. push notification
    summary = f"{etype.__name__}: {evalue}"
    notify(msg=summary, title="Python Error")

    # 2. let IPython show its normal traceback
    shell.showtraceback((etype, evalue, tb), tb_offset=tb_offset)
    return None            # ← must be None or a list of strings

get_ipython().set_custom_exc((Exception,), notify_exc)


## load data

## load standard data


In [ ]:
df = pd.read_pickle(resdir/'0506_m51df.pkl')
len(df)
df=df[df.session==41]

In [5]:
def process_one(y):
    related_taskvar=np.array(normalize_z(y))
    mask=(related_taskvar > -4) & (related_taskvar < 4)
    related_taskvar[~mask]=0
    return related_taskvar, mask

def process(rawtaskvar, returnmask=False):
    # normalize each item to 0 mean 1 std
    rawtaskvar=[np.array(normalize_z(y)) for y in rawtaskvar]
    related_taskvar=np.vstack(rawtaskvar).T
    # related_taskvar=np.clip(related_taskvar,-4,4)
    mask=(related_taskvar > -4) & (related_taskvar < 4)
    mask=np.all(mask, axis=1)
    related_taskvar=related_taskvar[mask]
    if returnmask:
        return related_taskvar, mask
    return related_taskvar
def process_list(rawtaskvar, returnmask=False):
    # normalize each item to 0 mean 1 std
    rawtaskvar=[np.array(normalize_z(y)) for y in rawtaskvar]
    mask=[(a > -4) & (a < 4) for a in rawtaskvar]
    mask=np.all(mask)
    related_taskvar=[a[mask] for a in rawtaskvar]
    if returnmask:
        return related_taskvar, mask
    return related_taskvar

In [6]:
# error distribution
def get_stop_error(row):
    return ((row.fx[-1]-row.mx[-1])**2+(row.fy[-1]-row.my[-1])**2)**0.5
df['state_error']=df.apply(get_stop_error, axis=1)
def get_stop_error(row):
    return ((row.fx[-1]-row.bmx[-1])**2+(row.fy[-1]-row.bmy[-1])**2)**0.5
df['belief_error']=df.apply(get_stop_error, axis=1)

# time ratio
df['time_ratio']=df.apply(lambda x: x.timer/x.timer[-1], axis=1)


In [7]:
for session in sorted(df.session.unique())[:1]:
    sessdf=df[(df.session==session)]

state=np.concatenate(sessdf.belief_ff_hori.to_numpy())
belief=np.concatenate(sessdf.ff_hori.to_numpy())
eye=np.concatenate(sessdf.eye_hori.to_numpy())
timeratio=np.concatenate(sessdf.time_ratio.to_numpy())
timer=np.concatenate(sessdf.timer.to_numpy())
countdown=np.concatenate(sessdf.countdown.to_numpy())


In [8]:
# make a df by time (each row is a time, instead of a trial)
sessdf['density_t']=sessdf.apply(lambda x: np.array([[x.density]*len(x.mx)]).reshape(-1), axis=1)
sessdf['fullon_t']=sessdf.apply(lambda x: np.array([[x.fullon]*len(x.mx)]).reshape(-1), axis=1)
sessdf['error_t']=sessdf.apply(lambda x: np.array([[x.error]*len(x.mx)]).reshape(-1), axis=1)

timedf=pd.DataFrame({'timer':timer})
timedf['rawcov']=list(np.vstack(sessdf['rawcov'].to_numpy()))
timedf['belief']=np.concatenate(sessdf.belief_ff_hori.to_numpy())
timedf['state']=np.concatenate(sessdf.ff_hori.to_numpy())
timedf['eye']=np.concatenate(sessdf.eye_hori.to_numpy())
timedf['timeratio']=np.concatenate(sessdf.time_ratio.to_numpy())
timedf['mv']=np.concatenate(sessdf.mv.to_numpy())
timedf['mw']=np.concatenate(sessdf.mw.to_numpy())
timedf['belief_heading']=np.concatenate(sessdf.belief_heading.to_numpy())
timedf['timer']=np.concatenate(sessdf.timer.to_numpy())
timedf['countdown']=np.concatenate(sessdf.countdown.to_numpy())
timedf['belief_angle_from_start']=np.concatenate(sessdf.belief_angle_from_start.to_numpy())
timedf['PPC']=list(np.vstack(sessdf.PPC.to_numpy()))

timedf['uncertainty']=list(np.concatenate(sessdf.relcov.to_numpy()))

timedf['density']=np.concatenate(sessdf.density_t.to_numpy())
timedf['fullon']=np.concatenate(sessdf.fullon_t.to_numpy())
timedf['error']=np.concatenate(sessdf.error_t.to_numpy())


timedf['dbelief']=np.concatenate(sessdf.apply(lambda x: np.diff(x.belief_ff_hori, prepend=x.belief_ff_hori[:1],axis=0),axis=1).to_numpy())
timedf['deye']=np.concatenate(sessdf.apply(lambda x: np.diff(x.eye_hori, prepend=x.eye_hori[:1],axis=0),axis=1).to_numpy())
timedf['dneural']=list(np.vstack(sessdf.apply(lambda x: np.diff(x.PPC, prepend=x.PPC[:1], axis=0),axis=1).to_numpy()))


timedf.head()
timedf = timedf.dropna(subset=['eye', 'state', 'belief'])

## load iti data

In [9]:
itidf = pd.read_pickle(resdir/'0421iti_m51df.pkl')
len(itidf)
itidf=itidf[itidf.session==41]
itidf['time_ratio']=itidf.apply(lambda x: x.timer/x.timer[-1], axis=1)

In [10]:
for session in sorted(itidf.session.unique())[:1]:
    sessitidf=itidf[(itidf.session==session)]

# state=np.concatenate(sessitidf.belief_ff_hori.to_numpy())
belief=np.concatenate(sessitidf.ff_hori.to_numpy())
eye=np.concatenate(sessitidf.eye_hori.to_numpy())
timeratio=np.concatenate(sessitidf.time_ratio.to_numpy())
timer=np.concatenate(sessitidf.timer.to_numpy())
countdown=np.concatenate(sessitidf.countdown.to_numpy())

# make a itidf by time (each row is a time, instead of a trial)
sessitidf['density_t']=sessitidf.apply(lambda x: np.array([[x.density]*len(x.mx)]).reshape(-1), axis=1)
sessitidf['fullon_t']=sessitidf.apply(lambda x: np.array([[x.fullon]*len(x.mx)]).reshape(-1), axis=1)
sessitidf['error_t']=sessitidf.apply(lambda x: np.array([[x.error]*len(x.mx)]).reshape(-1), axis=1)

timeitidf=pd.DataFrame({'timer':timer})
# timeitidf['rawcov']=list(np.vstack(sessitidf['rawcov'].to_numpy()))
# timeitidf['belief']=np.concatenate(sessitidf.belief_ff_hori.to_numpy())
timeitidf['state']=np.concatenate(sessitidf.ff_hori.to_numpy())
timeitidf['eye']=np.concatenate(sessitidf.eye_hori.to_numpy())
timeitidf['timeratio']=np.concatenate(sessitidf.time_ratio.to_numpy())
timeitidf['mv']=np.concatenate(sessitidf.mv.to_numpy())
timeitidf['mw']=np.concatenate(sessitidf.mw.to_numpy())
# timeitidf['belief_heading']=np.concatenate(sessitidf.belief_heading.to_numpy())
timeitidf['timer']=np.concatenate(sessitidf.timer.to_numpy())
timeitidf['countdown']=np.concatenate(sessitidf.countdown.to_numpy())
# timeitidf['belief_angle_from_start']=np.concatenate(sessitidf.belief_angle_from_start.to_numpy())
timeitidf['PPC']=list(np.vstack(sessitidf.PPC.to_numpy()))

# timeitidf['uncertainty']=list(np.concatenate(sessitidf.relcov.to_numpy()))

timeitidf['density']=np.concatenate(sessitidf.density_t.to_numpy())
timeitidf['fullon']=np.concatenate(sessitidf.fullon_t.to_numpy())
timeitidf['error']=np.concatenate(sessitidf.error_t.to_numpy())


# timeitidf['dbelief']=np.concatenate(sessitidf.apply(lambda x: np.diff(x.belief_ff_hori, prepend=x.belief_ff_hori[:1],axis=0),axis=1).to_numpy())
timeitidf['deye']=np.concatenate(sessitidf.apply(lambda x: np.diff(x.eye_hori, prepend=x.eye_hori[:1],axis=0),axis=1).to_numpy())
timeitidf['dneural']=list(np.vstack(sessitidf.apply(lambda x: np.diff(x.PPC, prepend=x.PPC[:1], axis=0),axis=1).to_numpy()))


timeitidf.head()
# timeitidf = timeitidf.dropna(subset=['eye', 'state'])

,timer,state,eye,timeratio,mv,mw,countdown,PPC,density,fullon,error,deye,dneural
0,0,-0.411339,28.940651,0.000000,-0.003190,-0.001378,-37,"[0.106192164, 0.19640955, 0.0, 0.0, 0.05650172...",0.005,0,174.501175,0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,1,-0.470667,28.920336,0.027027,0.023303,0.007235,-36,"[0.10775572, 0.1846899, 0.0, 3.3458742e-05, 0....",0.005,0,174.501175,-0.020315,"[0.0015635565, -0.011719659, 0.0, 3.3458742e-0..."
2,2,-0.533338,28.902283,0.054054,-0.004524,-0.002155,-35,"[0.109807715, 0.16330647, 0.0, 8.815204e-05, 0...",0.005,0,174.501175,-0.018053,"[0.0020519942, -0.02138342, 0.0, 5.4693297e-05..."
3,3,-0.500699,28.880598,0.081081,-0.015537,-0.000820,-34,"[0.110675946, 0.13578184, 0.0, 0.0002181784, 0...",0.005,0,174.501175,-0.021685,"[0.00086823106, -0.027524635, 0.0, 0.000130026..."
4,4,-0.481149,28.881540,0.108108,0.015418,0.008251,-33,"[0.10878229, 0.10615871, 0.0, 0.00050727994, 0...",0.005,0,174.501175,0.000942,"[-0.0018936545, -0.029623128, 0.0, 0.000289101..."


# done

In [13]:
neural_data = np.vstack(timedf['PPC'].to_numpy())
belief_data = timedf['belief'].to_numpy()
eye_data = timedf['eye'].to_numpy()


import numpy as np
from sklearn.linear_model import LassoCV
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

# Ensure all inputs are numpy arrays
X = neural_data  # [T, N]
y_belief = np.vstack(belief_data)  # [T, D1]
y_eye = np.vstack(eye_data)        # [T, D2]

# Optional: z-score inputs (standardizing helps Lasso)
X = (X - X.mean(0)) / X.std(0)

def crossval_lasso(X, Y, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=0)
    y_true_all, y_pred_all = [], []

    for train_idx, test_idx in kf.split(X):
        model = LassoCV(cv=5, max_iter=10000)
        model.fit(X[train_idx], Y[train_idx])
        y_pred = model.predict(X[test_idx])
        y_true_all.append(Y[test_idx])
        y_pred_all.append(y_pred)

    y_true_all = np.concatenate(y_true_all, axis=0)
    y_pred_all = np.concatenate(y_pred_all, axis=0)
    r2 = r2_score(y_true_all, y_pred_all)
    return r2

# Run for each prediction target
r2_belief = crossval_lasso(X, y_belief)
r2_eye = crossval_lasso(X, y_eye)

print(f'Cross-validated R² (neural → belief): {r2_belief:.3f}')
print(f'Cross-validated R² (neural → eye): {r2_eye:.3f}')


Cross-validated R² (neural → belief): 0.252
Cross-validated R² (neural → eye): 0.410


In [ ]:
# import numpy as np
# from sklearn.linear_model import RidgeCV
# from sklearn.model_selection import KFold
# from sklearn.metrics import r2_score
# from sklearn.preprocessing import StandardScaler

# # Inputs: 
# # neural_data: [T, N] - neural features (e.g., spike rates)
# # belief_data: [T, D1] - belief variables (can be 1D or 2D)
# # eye_data: [T, D2] - eye variables

# def standardize(X):
#     return (X - X.mean(0)) / X.std(0)

# def crossval_ridge(X, Y, n_splits=5, alphas=np.logspace(-3, 3, 10)):
#     kf = KFold(n_splits=n_splits, shuffle=True, random_state=0)
#     y_true_all, y_pred_all = [], []

#     for train_idx, test_idx in kf.split(X):
#         X_train, X_test = X[train_idx], X[test_idx]
#         Y_train, Y_test = Y[train_idx], Y[test_idx]

#         model = RidgeCV(alphas=alphas, store_cv_values=False)
#         model.fit(X_train, Y_train)

#         Y_pred = model.predict(X_test)
#         y_true_all.append(Y_test)
#         y_pred_all.append(Y_pred)

#     y_true_all = np.vstack(y_true_all)
#     y_pred_all = np.vstack(y_pred_all)

#     r2 = r2_score(y_true_all, y_pred_all, multioutput='uniform_average')
#     return r2

# # Example usage
# X = standardize(neural_data)               # [T, N]
# Y_belief = standardize(np.array(belief_data))  # [T, D1]
# Y_eye = standardize(np.array(eye_data))        # [T, D2]

# r2_belief = crossval_ridge(X, Y_belief)
# r2_eye = crossval_ridge(X, Y_eye)

# print(f'Cross-validated R² (neural → belief): {r2_belief:.3f}')
# print(f'Cross-validated R² (neural → eye): {r2_eye:.3f}')


ValueError: all the input array dimensions except for the concatenation axis must match exactly, but along dimension 1, the array at index 0 has size 80024 and the array at index 2 has size 80023

In [22]:
import numpy as np
import statsmodels.api as sm

# Inputs: [T, N] and [T, D]
neural_data = np.vstack(timedf['PPC'].to_numpy())        # [T, N]
eye_data = np.vstack(timedf['eye'].to_numpy())           # [T, 2]
belief_data = np.vstack(timedf['belief'].to_numpy())     # [T, D]

# Remove NaNs
X_eye = eye_data
X_belief = belief_data
X_all = np.hstack([X_eye, X_belief])
valid = np.isfinite(X_all).all(1) & np.isfinite(neural_data).all(1)

X_eye = X_eye[valid]
X_belief = X_belief[valid]
y_all = neural_data[valid]
X_all = np.hstack([X_eye, X_belief])
X_design_full = sm.add_constant(X_all)
X_design_eye = sm.add_constant(X_eye)
X_design_belief = sm.add_constant(X_belief)

# Store results
unique_eye_r2 = []
unique_belief_r2 = []
total_r2 = []

for i in range(y_all.shape[1]):
    y = y_all[:, i]

    # Full model: eye + belief
    model_full = sm.GLM(y, X_design_full, family=sm.families.Gaussian()).fit()
    pred_full = model_full.predict(X_design_full)
    r2_full = 1 - np.sum((y - pred_full)**2) / np.sum((y - np.mean(y))**2)

    # No eye: only belief
    model_belief = sm.GLM(y, X_design_belief, family=sm.families.Gaussian()).fit()
    pred_belief = model_belief.predict(X_design_belief)
    r2_belief = 1 - np.sum((y - pred_belief)**2) / np.sum((y - np.mean(y))**2)

    # No belief: only eye
    model_eye = sm.GLM(y, X_design_eye, family=sm.families.Gaussian()).fit()
    pred_eye = model_eye.predict(X_design_eye)
    r2_eye = 1 - np.sum((y - pred_eye)**2) / np.sum((y - np.mean(y))**2)

    # Partition
    unique_eye_r2.append(r2_full - r2_belief)
    unique_belief_r2.append(r2_full - r2_eye)
    total_r2.append(r2_full)

# Convert to array
unique_eye_r2 = np.array(unique_eye_r2)
unique_belief_r2 = np.array(unique_belief_r2)
total_r2 = np.array(total_r2)

# Print summary
print(f"Mean total R²: {np.mean(total_r2):.4f}")
print(f"Mean unique eye R²: {np.mean(unique_eye_r2):.4f}")
print(f"Mean unique belief R²: {np.mean(unique_belief_r2):.4f}")


Mean total R²: 0.0082
Mean unique eye R²: 0.0040
Mean unique belief R²: 0.0021


In [24]:
timedf.columns

Index(['timer', 'rawcov', 'belief', 'state', 'eye', 'timeratio', 'mv', 'mw',
       'belief_heading', 'countdown', 'belief_angle_from_start', 'PPC',
       'uncertainty', 'density', 'fullon', 'error', 'dbelief', 'deye',
       'dneural'],
      dtype='object')

In [ ]:
import numpy as np
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

# 1. Load predictors (each is shape [T, 1])
eye = np.vstack(timedf['eye'].to_numpy())          # [T, 1]
state = np.vstack(timedf['state'].to_numpy())      # [T, 1]
timer = timedf['timer'].to_numpy().reshape(-1, 1)  # [T, 1]
mw = timedf['mw'].to_numpy().reshape(-1, 1)        # [T, 1]
fr = np.vstack(timedf['PPC'].to_numpy())           # [T, N]

# 2. Add one lag of firing rate as spike history
lag = 1
fr_lagged = np.roll(fr, lag, axis=0)
fr_lagged[:lag, :] = np.nan  # mask first row

# 3. Build base design matrix: [T, 5]
X_base = np.hstack([eye, state, timer, mw, fr_lagged])

# 4. Remove rows with any NaNs
valid = np.isfinite(X_base).all(axis=1) & np.isfinite(fr).all(axis=1)
X_base = X_base[valid]
fr = fr[valid]

# 5. Setup RidgeCV
alphas = np.logspace(-3, 3, 10)
kf = KFold(n_splits=5, shuffle=True, random_state=0)
n_neurons = fr.shape[1]
r2s = []

# 6. Loop over neurons
for i in range(n_neurons):
    print(f"Neuron {i+1}/{n_neurons}...")
    y = fr[:, i]
    y_pred_all, y_true_all = [], []

    for train_idx, test_idx in kf.split(X_base):
        ridge = RidgeCV(alphas=alphas)
        ridge.fit(X_base[train_idx], y[train_idx])
        y_pred = ridge.predict(X_base[test_idx])
        y_pred_all.append(y_pred)
        y_true_all.append(y[test_idx])

    y_pred_all = np.concatenate(y_pred_all)
    y_true_all = np.concatenate(y_true_all)
    r2 = r2_score(y_true_all, y_pred_all)
    r2s.append(r2)

# 7. Report result
print(f"\nModel 1 (Base): Mean cross-validated pseudo-R² across neurons = {np.mean(r2s):.4f}")


Neuron 1/112...
Neuron 2/112...
Neuron 3/112...
Neuron 4/112...


KeyboardInterrupt: 

In [ ]:
# fix alpha ridge, faster
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

# 1. Load predictors (each is shape [T, 1])
eye = np.vstack(timedf['eye'].to_numpy())          
state = np.vstack(timedf['state'].to_numpy())      
timer = timedf['timer'].to_numpy().reshape(-1, 1)  
mw = timedf['mw'].to_numpy().reshape(-1, 1)        
fr = np.vstack(timedf['PPC'].to_numpy())           

# 2. Spike history (1 lag)
lag = 1
fr_lagged = np.roll(fr, lag, axis=0)
fr_lagged[:lag, :] = np.nan

# 3. Design matrix
X_base = np.hstack([eye, state, timer, mw, fr_lagged])
valid = np.isfinite(X_base).all(axis=1) & np.isfinite(fr).all(axis=1)
X_base = X_base[valid]
fr = fr[valid]

# 4. Fixed-alpha ridge
ridge = Ridge(alpha=10.0)
kf = KFold(n_splits=5, shuffle=True, random_state=0)
r2s = []

# 5. Per-neuron loop
for i in range(fr.shape[1]):
    print(f"Neuron {i+1}/{fr.shape[1]}...")
    y = fr[:, i]
    y_true_all, y_pred_all = [], []

    for train_idx, test_idx in kf.split(X_base):
        ridge.fit(X_base[train_idx], y[train_idx])
        y_pred = ridge.predict(X_base[test_idx])
        y_pred_all.append(y_pred)
        y_true_all.append(y[test_idx])

    y_true_all = np.concatenate(y_true_all)
    y_pred_all = np.concatenate(y_pred_all)
    r2 = r2_score(y_true_all, y_pred_all)
    r2s.append(r2)

print(f"\nModel 1 (Base): Mean cross-validated pseudo-R² = {np.mean(r2s):.4f}")


Neuron 1/112...
Neuron 2/112...
Neuron 3/112...
Neuron 4/112...
Neuron 5/112...
Neuron 6/112...
Neuron 7/112...
Neuron 8/112...
Neuron 9/112...
Neuron 10/112...
Neuron 11/112...
Neuron 12/112...
Neuron 13/112...
Neuron 14/112...
Neuron 15/112...
Neuron 16/112...
Neuron 17/112...
Neuron 18/112...
Neuron 19/112...
Neuron 20/112...
Neuron 21/112...
Neuron 22/112...
Neuron 23/112...
Neuron 24/112...
Neuron 25/112...
Neuron 26/112...
Neuron 27/112...
Neuron 28/112...
Neuron 29/112...
Neuron 30/112...
Neuron 31/112...
Neuron 32/112...
Neuron 33/112...
Neuron 34/112...
Neuron 35/112...
Neuron 36/112...
Neuron 37/112...
Neuron 38/112...
Neuron 39/112...
Neuron 40/112...
Neuron 41/112...
Neuron 42/112...
Neuron 43/112...
Neuron 44/112...
Neuron 45/112...
Neuron 46/112...
Neuron 47/112...
Neuron 48/112...
Neuron 49/112...
Neuron 50/112...
Neuron 51/112...
Neuron 52/112...
Neuron 53/112...
Neuron 54/112...
Neuron 55/112...
Neuron 56/112...
Neuron 57/112...
Neuron 58/112...
Neuron 59/112...
Neuron

# plots

In [ ]:
notify('done')